### RAG Pipeline - Data Ingestion to Vector DB Pipeline


In [3]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path



In [4]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")


Found 3 PDF files to process

Processing: CC & DC – U & S V1.pdf
  ✓ Loaded 6 pages

Processing: leave.pdf
  ✓ Loaded 11 pages

Processing: Website Privacy Policy V2.pdf
  ✓ Loaded 8 pages

Total documents loaded: 25


In [5]:
### Text Splitting get into chunks 

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into chunks"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    if split_docs:
        print(f"\n Example chunk:")
        print(f"Content :{split_docs[0].page_content[:200]}")  # Print first 200 characters of the first chunk
        print(f"Metadata :{split_docs[0].metadata}")  # Print metadata of the first chunk
    
    return split_docs

# Split the loaded PDF documents into chunks





    

In [6]:
chunks = split_documents(all_pdf_documents)

Split 25 documents into 63 chunks

 Example chunk:
Content :Confidential property of Consint.ai. Do not distribute or reproduce without express permission from Consint.ai 
 
Corporate Credit and Debit Cards 
Usage and Settlement
Metadata :{'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2026-04-17T17:38:37+05:30', 'author': 'pc', 'moddate': '2026-04-17T17:38:37+05:30', 'source': '..\\data\\pdf_files\\CC & DC – U & S V1.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1', 'source_file': 'CC & DC – U & S V1.pdf', 'file_type': 'pdf'}


In [7]:
chunks

[Document(metadata={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2026-04-17T17:38:37+05:30', 'author': 'pc', 'moddate': '2026-04-17T17:38:37+05:30', 'source': '..\\data\\pdf_files\\CC & DC – U & S V1.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1', 'source_file': 'CC & DC – U & S V1.pdf', 'file_type': 'pdf'}, page_content='Confidential property of Consint.ai. Do not distribute or reproduce without express permission from Consint.ai \n \nCorporate Credit and Debit Cards \nUsage and Settlement'),
 Document(metadata={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2026-04-17T17:38:37+05:30', 'author': 'pc', 'moddate': '2026-04-17T17:38:37+05:30', 'source': '..\\data\\pdf_files\\CC & DC – U & S V1.pdf', 'total_pages': 6, 'page': 1, 'page_label': '2', 'source_file': 'CC & DC – U & S V1.pdf', 'file_type': 'pdf'}, page_content='Confidential property of Consint.ai. Do not distribute or reproduce without exp

### embedding and vector store db

In [8]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity


In [9]:
class EmbeddingManager:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager with a specified model name and load the model.

        Args:
            model_name (str): The name of the sentence transformer model to use for generating embeddings.
            Hugging face model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()
    
    def _load_model(self):
        """Load the sentence transformer model."""
        try:
            self.model = SentenceTransformer(self.model_name)
            print(f"Model '{self.model_name}' loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model '{self.model_name}': {e}")
            raise
    
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts.

        Args:
            texts (List[str]): A list of strings to generate embeddings for.
        Returns:
            np.ndarray: An array of embeddings corresponding to the input texts.
        """
        if not self.model:
            raise ValueError("Model not loaded. Call _load_model() first.")
        
        print(f"Generating embeddings for {len(texts)} texts...") 
        embeddings = self.model.encode(texts,show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    

    

In [10]:
embedding_manager = EmbeddingManager()
embedding_manager

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3736.27it/s]


Model 'all-MiniLM-L6-v2' loaded successfully. Embedding dimension: 384


### Class Vector Store

In [11]:
class VectorStore:
    def __init__(self,collection_name:str="pdf_documents",persist_directory:str="../data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()
    
    def _initialize_store(self):
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            self.collection = self.client.get_or_create_collection(name=self.collection_name,metadata={"description": "PDF Document for Embeddings"})
            print(f"Vector store initialized at '{self.persist_directory}' with collection '{self.collection_name}'")
            print(f"Collection metadata: {self.collection.metadata} with count of : {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
    
    def add_documents(self,documents:List[Any],embeddings:np.ndarray):
        """
        Add documents and their corresponding embeddings to the vector store.

        Args:
            documents (List[Any]): A list of document objects containing metadata.
            embeddings (np.ndarray): An array of embeddings corresponding to the documents.
        """
        if len(documents) != len(embeddings):
            raise ValueError("The number of documents must match the number of embeddings.")
        if not self.collection:
            raise ValueError("Vector store not initialized. Call _initialize_store() first.")
        
        print(f"Adding {len(documents)} documents to vector store...")

        # Prepare data for chroma 

        ids=[]
        metadatas=[]
        documents_texts=[]
        embeddings_list=[]

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # Document Content
            documents_texts.append(doc.page_content)

            # Embeddings 
            embeddings_list.append(embedding.tolist())

            # Add to chroma collection
            try:            
                self.collection.add(
                    ids=[doc_id],
                    embeddings=[embedding.tolist()],
                    metadatas=[metadata],
                    documents=[doc.page_content]
                )
                print(f"  ✓ Added document ID: {doc_id} with metadata: {metadata}")
            except Exception as e:
                print(f"  ✗ Error adding document ID: {doc_id} - {e}")
                raise

vectorStore = VectorStore()  

Vector store initialized at '../data/vector_store' with collection 'pdf_documents'
Collection metadata: {'description': 'PDF Document for Embeddings'} with count of : 252


In [12]:
# Convert the text to embeddings
texts = [doc.page_content for doc in chunks]

# Generate the embeddings
embeddings = embedding_manager.generate_embeddings(texts)

# Add the documents and embeddings to the vector store
vectorStore.add_documents(chunks,embeddings)


Generating embeddings for 63 texts...


Batches: 100%|██████████| 2/2 [00:02<00:00,  1.31s/it]


Generated embeddings with shape: (63, 384)
Adding 63 documents to vector store...
  ✓ Added document ID: doc_3e4b8788_0 with metadata: {'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2026-04-17T17:38:37+05:30', 'author': 'pc', 'moddate': '2026-04-17T17:38:37+05:30', 'source': '..\\data\\pdf_files\\CC & DC – U & S V1.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1', 'source_file': 'CC & DC – U & S V1.pdf', 'file_type': 'pdf', 'doc_index': 0, 'content_length': 168}
  ✓ Added document ID: doc_4f61a21a_1 with metadata: {'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2026-04-17T17:38:37+05:30', 'author': 'pc', 'moddate': '2026-04-17T17:38:37+05:30', 'source': '..\\data\\pdf_files\\CC & DC – U & S V1.pdf', 'total_pages': 6, 'page': 1, 'page_label': '2', 'source_file': 'CC & DC – U & S V1.pdf', 'file_type': 'pdf', 'doc_index': 1, 'content_length': 949}
  ✓ Added document ID: doc_1a1a9438_2 with metadata: {'p

### Create Retreiver Pipeline from vector store

In [13]:

class RAGRetriever:
    def __init__(self,vector_store:VectorStore,embedding_manager:EmbeddingManager):
        """ 
        Initialize the retriever

        Args:
            vector_store (VectorStore): An instance of the VectorStore class to retrieve documents from.
            embedding_manager (EmbeddingManager): An instance of the EmbeddingManager class to generate query embeddings.
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
    
    def retrieve(self,query:str,top_k:int=5,score_threshold:float=0.0) -> List[Dict[str,Any]]:
        """
        Retrieve relevant documents from the vector store based on a query.

        Args:
            query (str): The input query string to search for relevant documents.
            top_k (int): The number of top relevant documents to retrieve.
            score_threshold (float): The minimum similarity score for retrieved documents.
        """

        print(f"Retrieving documents for query: '{query}' with top_k={top_k} and score_threshold={score_threshold}")

        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # Search in the vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k,
            )
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
            
            for i,(doc_id,document,metadata,distance) in enumerate(zip(ids,documents,metadatas,distances)):
                similarity_score = 1 - distance
                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "content": document,
                        "metadata": metadata,
                        "distance": distance,
                        "similarity_score": similarity_score,
                        "rank": i + 1
                    })
        except Exception as e:
            print(f"Error occurred while retrieving documents: {e}")

        return retrieved_docs

rag_retriever = RAGRetriever(vectorStore, embedding_manager)



    


In [14]:
rag_retriever.retrieve("When should I take Privileged leave?")

Retrieving documents for query: 'When should I take Privileged leave?' with top_k=5 and score_threshold=0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 37.74it/s]

Generated embeddings with shape: (1, 384)


[{'id': 'doc_10bba9b4_22',
  'content': 'requirement) during current calendar year only. \n \n6 Privilege Leave \nThe company strongly encourages employees to use privilege leave for vacation to ensure a \nhealthy work life balance. \n \n6.1 Employees are expected to plan their privilege leave at least 14 (fourteen) days in advance \nof taking the leave. Privilege leave can be taken only with prior approval of the line  \nmanager. \n6.2 Privilege leave carried forward from the previous year and those earned during a calendar \nyear can be accumulated and utilized any time during the same  calendar year. Any \nunutilized leave more than 30 (thirty) days will lapse at the end of the calendar year and \nonly 30 (thirty) days shall be carried forward to the next calendar year. \n6.3 Encashment of accumulated privilege leaves is available only upon separation  from the \ncompany and the same will be up to a maximum of 30 days.',
  'metadata': {'doc_index': 22,
   'author': 'pc',
   'moddate

In [15]:
rag_retriever.retrieve("What is Unauthorized Absence")

Retrieving documents for query: 'What is Unauthorized Absence' with top_k=5 and score_threshold=0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 48.62it/s]

Generated embeddings with shape: (1, 384)


[{'id': 'doc_f30f377a_37',
  'content': 'without pay (LWP) period. For instance, if the employee takes a Friday- Monday leave and \nthis or her leave balance is exhausted, all four (4) days including the weekly offs \n(Saturday and Sunday) will be marked as leave without pay. \n14. Unauthorized Absence \n14.1 Unauthorized absence is when an employee is absent form work without communicating \nthe reason for absence. \n14.2 Employees are required to communicate reason for absence within 2 business days from \nthe first day of their absence. \n14.3 Employees must take prior approval from their line manager if they wish to be absent \nfrom work during the normal working hours. \n14.4 Such unauthorized absence will be considered as leave without pay and failure to notify \nthe relevant line manager of a period of absence my result in disciplinary action in line \nwith the Disciplinary, Capability and Grievance Policy.',
  'metadata': {'total_pages': 11,
   'content_length': 874,
   'author

In [16]:

rag_retriever.retrieve("Type of Maternity Leaves")

Retrieving documents for query: 'Type of Maternity Leaves' with top_k=5 and score_threshold=0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 28.92it/s]

Generated embeddings with shape: (1, 384)


[{'id': 'doc_2dd12e0d_27',
  'content': 'delivery or child is handed over to the commissioning/adopting mother, or date of \nmiscarriage/medical termination are eligible for paid maternity leave. For employees with less \nthan 80 (eighty) days service, maternity leave will be treated as unpaid leave. All eligible women \nemployees are entitled to maternity leave, as shown in the table below. The maternity leave \nis inclusive of week offs, and public and national holidays. \n \nType of Maternity \nLeaves \nLeave \nEntitlement (In \nWeeks) \nDocuments required to be \nsubmitted to HR to Avail \nthe Leave \nLeave Commencement \nMaternity Leave in \ncase of women \nemployee up to \ntwo \n \n \n26 \n1. Confirmation of pregnancy \nalong with the date of \ndelivery. \nNot earlier than \neight (8) weeks prior \n(2) surviving \nchildren \n 2. Medical \ncertificate from \ncertified medical \npractitioner. \nto the date\n of delivery. \nMaternity Leave in \ncase of women \nemployee with two \n(2

### Integrate Vectordb Context with LLM output

In [19]:
from langchain_cohere import ChatCohere
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Initialize the Cohere model
cohere_api_key = os.getenv("COHERE_API_KEY")
if not cohere_api_key:
    raise ValueError("Cohere API key not found. Please set the COHERE_API_KEY environment variable.")

llm = ChatCohere(
    cohere_api_key=cohere_api_key,
    model="command-a-03-2025",
    temperature=0.7,
    max_tokens=1024,
)

print(f"✓ Cohere LLM initialized with model: command-a-03-2025")

### Simple RAG function to generate answer from retrieved documents
def generate_answer(query: str, retriever: RAGRetriever, llm, top_k: int = 5):
    retrieved_docs = retriever.retrieve(query, top_k=top_k)
    if not retrieved_docs:
        return "No relevant documents found to answer the query."

    # Combine retrieved documents into a single context, limiting to first 2 docs and 400 chars each
    context_parts = []
    for doc in retrieved_docs[:2]:  # Use only first 2 documents
        content = doc['content'][:400]  # Limit content to 400 characters
        if len(doc['content']) > 400:
            content += "..."
        context_parts.append(f"Document {doc['rank']} (Similarity: {doc['similarity_score']:.2f}):\n{content}")
    
    context = "\n\n".join(context_parts)

    # Create a simple prompt
    prompt = f"Based on this context, answer the question.\n\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"

    # Generate an answer using the Cohere model
    try:
        response = llm.invoke(prompt)
        # Handle different response types
        if hasattr(response, 'content'):
            return response.content.strip()
        else:
            return str(response).strip()
    except Exception as e:
        print(f"Error generating answer: {type(e).__name__}: {e}")
        return f"An error occurred while generating the answer."

✓ Cohere LLM initialized with model: command-a-03-2025


In [24]:
generate_answer("What is website security?", rag_retriever, llm)

Retrieving documents for query: 'What is website security?' with top_k=5 and score_threshold=0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 52.74it/s]

Generated embeddings with shape: (1, 384)


'Website security refers to the measures and protocols put in place to protect a website, its data, and its users from potential threats, attacks, and unauthorized access. Based on the context provided, key components of website security include:\n\n1. **Secure hosting infrastructure**: Ensuring the servers and systems hosting the website are protected against vulnerabilities and attacks.  \n2. **Monitoring and periodic security reviews**: Regularly assessing the website for potential security risks and ensuring ongoing protection.  \n3. **Compliance with legal and regulatory requirements**: Implementing safeguards that meet industry standards and laws to protect personal data and prevent breaches.  \n4. **Prevention of unauthorized access, misuse, modification, or disclosure**: Employing measures to safeguard data integrity and confidentiality.  \n5. **Controlled disclosure of information to third parties**: Sharing data only for legitimate purposes and with trusted entities like serv